# Bagheria 2026 — thread Mobilità

Il terzo focus del bando: **«Dinamiche e ruolo del pendolarismo verso Palermo»**.

**Prerequisiti**: `pipeline.fetch` e `pipeline.build` già eseguiti. Le definizioni condivise
(territori di confronto, anno base, trappole sui totali) stanno in `CLAUDE.md` e non si
ridefiniscono qui.

## Perché questo notebook esiste

La relazione del progetto, alla sezione 5 e di nuovo fra le cose che non afferma, dichiara:

> «nessuna fonte disponibile identifica Palermo come destinazione, quindi il *pendolarismo
> verso Palermo* della locandina non è misurabile con i dati pubblici».

Era vero per le fonti allora usate — il censimento permanente pubblica il pendolarismo
comunale solo come `dentro`/`fuori comune` (verificato: la dimensione `LOC_DEST` del
dataflow `DF_DCSS_ISTR_LAV_PEN_2_TV_5` è servita solo come `ALL`, `SMPUR`, `OMPUR`).

**Non è vero in generale.** ISTAT pubblica le *matrici del pendolarismo*: la matrice
origine-destinazione comune per comune, con sesso, motivo, mezzo, fascia oraria di partenza
e durata del tragitto (censimenti 1991, 2001, 2011) e, per il solo lavoro, rifatta sul
censimento permanente 2021. Questo notebook la usa e misura ciò che mancava.

## Cosa c'è dentro, in breve

1. la fonte e la sua **verifica** — la matrice ricostruisce sei indicatori 8milaCensus già pubblicati;
2. **dove vanno** i pendolari di Bagheria, 2011 e 2021, per nome del comune;
3. «Bagheria si muove poco» è un **effetto della taglia**: a parità di distanza e dimensione è nella media;
4. il **ribaltamento**: si esce per studiare, non per lavorare — e ha un genere;
5. **come** ci si va: il treno è il canale femminile;
6. **quando** si parte;
7. un meccanismo che i dati **non** sostengono, riportato perché è stato testato;
8. l'**ultimo miglio** a Palermo, dai dati del Comune (fonte del bando);
9. il bersaglio, in persone.

## Fonti, e il vincolo del bando

Il bando indica tre fonti: ISTAT (8milaCensus), Open Data Sicilia, Comune di Palermo.
Qui si usano: **8milaCensus** (indicatori `M1`-`M9`, come da bando), la **matrice del
pendolarismo ISTAT** — che è la microdato da cui quegli stessi indicatori sono calcolati,
e questo notebook lo dimostra numericamente — e il **GTFS AMAT del Comune di Palermo**
(§8). Open Data Sicilia non ha dataset di mobilità: il catalogo CKAN risponde con quattro
pacchetti su «mobilità», tutti di finanza pubblica, e zero su «pendolarismo» (ricognizione
del 2026-08-29, `docs/sources.md` §12).

# **Caricamento**

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

RADICE = Path.cwd()
if RADICE.name == "notebooks":
    RADICE = RADICE.parent
PROCESSED = RADICE / "data" / "processed"

BAGHERIA, PALERMO = "082006", "082053"
# I cinque comuni geograficamente più vicini, gli stessi del thread genere.
VICINI = ["082067", "082035", "082079", "082023", "082048"]

od = pd.read_csv(PROCESSED / "pendolarismo_od_long.csv", dtype={"origine": str, "destinazione": str})
mezzo = pd.read_csv(PROCESSED / "pendolarismo_mezzo_long.csv", dtype={"origine": str})
benchmark = pd.read_csv(PROCESSED / "pendolarismo_benchmark_long.csv")
mezzi = pd.read_csv(PROCESSED / "pendolarismo_mezzi.csv")
territori = pd.read_csv(PROCESSED / "territori.csv", dtype={"territorio": str})
centroidi = pd.read_csv(PROCESSED / "comuni_sicilia_centroidi.csv", dtype={"territorio": str})
ottomila = pd.read_csv(PROCESSED / "ottomilacensus_long.csv", dtype={"territorio": str})

NOMI = dict(zip(territori["territorio"], territori["nome_territorio"]))
# L'universo dei confronti cross-comunali: i 390 comuni siciliani ai confini 2011, lo stesso
# denominatore degli altri due thread (Misiliscemi, nato nel 2021, resta fuori).
COMUNI390 = sorted(ottomila.loc[ottomila["livello"].eq(1), "territorio"].unique())
assert len(COMUNI390) == 390, len(COMUNI390)

print(f"matrice OD: {len(od):,} righe, anni {sorted(od['anno'].unique())}")
print(f"mezzo/orario/durata: {len(mezzo):,} righe (solo 2011)")
print(f"comuni nel panel: {len(COMUNI390)}")

matrice OD: 76,222 righe, anni [np.int64(2001), np.int64(2011), np.int64(2021)]
mezzo/orario/durata: 98,201 righe (solo 2011)
comuni nel panel: 390


# **1. La fonte, e come si controlla che sia letta bene**

Il tracciato è a campi fissi e senza intestazione: un campo sfalsato produrrebbe numeri
plausibili e sbagliati. Il controllo non è stilistico ma sostanziale — **8milaCensus
pubblica nove indicatori di mobilità `M1`-`M9` calcolati su questa stessa rilevazione**,
quindi la matrice deve ricostruirli. Se tornano, il tracciato è letto giusto.

`M1` e `M2` non sono ricostruibili: hanno al denominatore la popolazione fino a 64 anni,
che nella matrice non c'è. Gli altri sette sì.

In [2]:
def indicatore_pubblicato(codice: str, territorio: str = BAGHERIA, anno: int = 2011) -> float:
    riga = ottomila[ottomila["territorio"].eq(territorio) & ottomila["anno"].eq(anno)
                    & ottomila["indicatore"].eq(codice)]
    return riga["valore"].iloc[0]


bag_od = od[od["anno"].eq(2011) & od["origine"].eq(BAGHERIA)]
per_motivo = bag_od.pivot_table(index="motivo", columns="luogo", values="persone", aggfunc="sum")

bag_mezzo = mezzo[mezzo["origine"].eq(BAGHERIA)]
classe = bag_mezzo["mezzo"].map(dict(zip(mezzi["mezzo"], mezzi["classe"])))
totale_stima = bag_mezzo["stima"].sum()
quota = lambda selezione: 100 * bag_mezzo.loc[selezione, "stima"].sum() / totale_stima

ricostruiti = {
    # M3/M4 = fuori comune su dentro comune, dentro un solo motivo.
    "M3": 100 * per_motivo.loc["lavoro", "fuori"] / per_motivo.loc["lavoro", "dentro"],
    "M4": 100 * per_motivo.loc["studio", "fuori"] / per_motivo.loc["studio", "dentro"],
    "M5": quota(classe.eq("privato a motore")),
    "M6": quota(classe.eq("collettivo")),
    "M7": quota(classe.eq("piedi o bici")),
    "M8": quota(bag_mezzo["durata"].isin(["fino a 15 min", "16-30 min"])),
    "M9": quota(bag_mezzo["durata"].eq("oltre 60 min")),
}
controllo = pd.DataFrame({
    "ricostruito dalla matrice": pd.Series(ricostruiti).round(1),
    "pubblicato da 8milaCensus": pd.Series({c: indicatore_pubblicato(c) for c in ricostruiti}),
})
controllo["coincide"] = controllo.iloc[:, 0].eq(controllo.iloc[:, 1])
print(controllo.to_string())
print()
print("Totali nazionali dichiarati nei leggimi ISTAT contro quelli ricostruiti:")
for anno, atteso in [(2011, 28_871_447), (2021, 19_565_808)]:
    nostro = benchmark.query("anno == @anno and territorio == 'IT'")["persone"].sum()
    print(f"  {anno}: {nostro:>12,}  atteso {atteso:>12,}  {'ok' if nostro == atteso else 'DIVERSO'}")
assert controllo["coincide"].all(), "il tracciato non è letto correttamente"

    ricostruito dalla matrice  pubblicato da 8milaCensus  coincide
M3                       76.1                       76.1      True
M4                       19.6                       19.6      True
M5                       65.2                       65.2      True
M6                        8.4                        8.4      True
M7                       25.7                       25.7      True
M8                       83.3                       83.3      True
M9                        3.6                        3.6      True

Totali nazionali dichiarati nei leggimi ISTAT contro quelli ricostruiti:
  2011:   28,871,447  atteso   28,871,447  ok
  2021:   19,565,808  atteso   19,565,808  ok


**📌 Risultato chiave** — Sette indicatori su sette coincidono alla prima cifra decimale con
quelli pubblicati, e i due totali nazionali con quelli dichiarati da ISTAT. Il tracciato è
letto correttamente, e — punto non secondario per il vincolo sulle fonti del bando — questo
**dimostra che la matrice e 8milaCensus sono la stessa rilevazione**: la matrice è il livello
sottostante da cui gli `M1`-`M9` sono calcolati, non una fonte alternativa.

Una nota sulla definizione di `M6`, che si scopre proprio da questo controllo: la «mobilità
pubblica» di 8milaCensus **esclude** l'autobus aziendale o scolastico (codice 06). Includendolo
verrebbe 8,9 invece di 8,4. Nel resto del notebook si usa la stessa convenzione di ISTAT.

# **2. Il pendolarismo verso Palermo, con il nome del comune**

La domanda del bando, alla lettera. La matrice dà il comune di destinazione, quindi la
risposta è diretta: fra chi esce da Bagheria, quanti vanno a Palermo e quanti altrove.

Due anni, e **due definizioni diverse**: il 2011 conta chi si sposta *giornalmente*, il 2021
chi si reca al lavoro *almeno tre giorni a settimana*. I livelli non stanno in serie — la
colonna `definizione` della tabella lo porta scritto in ogni riga — mentre la *composizione*
(dove vanno, su cento che escono) è molto più robusta al cambio.

In [3]:
def flussi(anno: int, motivo: str, origine: str = BAGHERIA) -> pd.DataFrame:
    d = od[od["anno"].eq(anno) & od["motivo"].eq(motivo) & od["origine"].eq(origine)]
    fuori = d[d["luogo"].eq("fuori")]
    t = (fuori.groupby("destinazione")["persone"].sum().sort_values(ascending=False)
         .rename("persone").to_frame())
    t["quota_su_chi_esce"] = 100 * t["persone"] / t["persone"].sum()
    t.insert(0, "comune", [NOMI.get(i, i) for i in t.index])
    return t


for anno, motivo in [(2011, "studio"), (2011, "lavoro"), (2021, "lavoro")]:
    t = flussi(anno, motivo)
    d = od[od["anno"].eq(anno) & od["motivo"].eq(motivo) & od["origine"].eq(BAGHERIA)]
    dentro = d.loc[d["luogo"].eq("dentro"), "persone"].sum()
    print(f"=== {anno}, {motivo} — {dentro:,} restano a Bagheria, {t['persone'].sum():,} escono "
          f"({100 * t['persone'].sum() / (dentro + t['persone'].sum()):.1f}%)")
    print(t.head(5).round(1).to_string(index=False))
    print()

=== 2011, studio — 8,726 restano a Bagheria, 1,706 escono (16.4%)
         comune  persone  quota_su_chi_esce
        Palermo     1554               91.1
   Santa Flavia       34                2.0
      Ficarazzi       31                1.8
Termini Imerese       18                1.1
   Casteldaccia       18                1.1

=== 2011, lavoro — 6,335 restano a Bagheria, 4,818 escono (43.2%)
         comune  persone  quota_su_chi_esce
        Palermo     3250               67.5
   Santa Flavia      330                6.8
Termini Imerese      243                5.0
   Casteldaccia      181                3.8
      Ficarazzi      133                2.8

=== 2021, lavoro — 7,166 restano a Bagheria, 4,808 escono (40.2%)
         comune  persone  quota_su_chi_esce
        Palermo     3129               65.1
   Santa Flavia      353                7.3
Termini Imerese      192                4.0
   Casteldaccia      168                3.5
      Ficarazzi      155                3.2



In [4]:
# Quanto è concentrata questa destinazione, rispetto agli altri comuni siciliani? La misura
# comparabile non è "quanti vanno a Palermo" (per un comune di Ragusa è zero per costruzione)
# ma "quanti vanno nel PROPRIO capoluogo di provincia".
CAPOLUOGHI = {"081": "081021", "082": "082053", "083": "083048", "084": "084001", "085": "085004",
              "086": "086009", "087": "087015", "088": "088009", "089": "089017"}
xy = centroidi.set_index("territorio")[["x", "y"]]


def quota_capoluogo(anno: int, motivo: str) -> pd.Series:
    d = od[od["anno"].eq(anno) & od["motivo"].eq(motivo)
           & od["origine"].isin(COMUNI390) & od["luogo"].eq("fuori")]
    capoluogo = d["origine"].str[:3].map(CAPOLUOGHI)
    verso = d[d["destinazione"].eq(capoluogo)].groupby("origine")["persone"].sum()
    totale = d.groupby("origine")["persone"].sum()
    return (100 * verso.reindex(totale.index).fillna(0) / totale).rename(f"{anno} {motivo}")


concentrazione = pd.concat([quota_capoluogo(2011, "studio"), quota_capoluogo(2011, "lavoro"),
                            quota_capoluogo(2021, "lavoro")], axis=1)
# I nove capoluoghi vanno esclusi: per loro la misura non è definita.
concentrazione = concentrazione.drop(index=[c for c in CAPOLUOGHI.values() if c in concentrazione.index])
riassunto = pd.DataFrame({
    "Bagheria": concentrazione.loc[BAGHERIA].round(1),
    "mediana dei 381 comuni": concentrazione.median().round(1),
    "percentile di Bagheria": [round(100 * (concentrazione[c] < concentrazione.loc[BAGHERIA, c]).mean())
                               for c in concentrazione.columns],
})
print(riassunto.to_string())

             Bagheria  mediana dei 381 comuni  percentile di Bagheria
2011 studio      91.1                    23.2                      97
2011 lavoro      67.5                    20.3                      93
2021 lavoro      65.1                    27.1                      94


**📌 Risultato chiave** — Il pendolarismo di Bagheria **è** pendolarismo verso Palermo, e in
misura fuori scala: nove studenti su dieci che escono dal comune vanno a Palermo (91,1% nel
2011) e due lavoratori su tre (67,5% nel 2011, 65,1% nel 2021), contro una mediana siciliana
attorno alla metà. Su tutte e tre le misure Bagheria sta oltre il **90° percentile** dei 381
comuni non capoluogo.

Il secondo comune di destinazione è Santa Flavia con il 6,8% (lavoro 2011): non esiste una
seconda direzione. **Bagheria ha un solo mercato del lavoro esterno, e si chiama Palermo.**
Questo è il fatto che rende la mobilità una leva di policy e non un dettaglio descrittivo:
non c'è da scegliere quale destinazione servire.

# **3. «Bagheria si muove poco»: è vero, ma è un effetto della taglia**

Il dato grezzo dice questo: `M2` (mobilità fuori comune) al 25° percentile dei 390 comuni,
`M4` (mobilità studentesca) al 18°. Guardando i vicini l'anomalia sembra clamorosa —
Ficarazzi manda fuori il 75% dei suoi pendolari, Bagheria il 40%.

Ma **fuori comune si va per mancanza di lavoro dentro**, e Bagheria è il comune più grande
della corona: 12.000 pendolari contro i 3.000 di Ficarazzi. La domanda giusta non è «quanto
esce» ma «quanto esce *rispetto a un comune della sua taglia e della sua distanza*».

In [5]:
def panel(anno: int, motivo: str) -> pd.DataFrame:
    d = od[od["anno"].eq(anno) & od["motivo"].eq(motivo) & od["origine"].isin(COMUNI390)]
    g = d.pivot_table(index="origine", columns="luogo", values="persone", aggfunc="sum").fillna(0)
    verso_pa = d[d["luogo"].eq("fuori") & d["destinazione"].eq(PALERMO)].groupby("origine")["persone"].sum()
    entrano = (od[od["anno"].eq(anno) & od["motivo"].eq(motivo) & od["destinazione"].isin(COMUNI390)
                  & od["origine"].ne(od["destinazione"])].groupby("destinazione")["persone"].sum())
    p = pd.DataFrame({"dentro": g.get("dentro", 0), "fuori": g.get("fuori", 0),
                      "verso_palermo": verso_pa.reindex(g.index).fillna(0),
                      "entrano": entrano.reindex(g.index).fillna(0)})
    p["pendolari"] = p["dentro"] + p["fuori"]
    p["quota_fuori"] = 100 * p["fuori"] / p["pendolari"]
    p["capoluogo"] = pd.Series(p.index, index=p.index).str[:3].map(CAPOLUOGHI)
    p["km_capoluogo"] = [np.hypot(*(xy.loc[t] - xy.loc[c])) / 1000 for t, c in zip(p.index, p["capoluogo"])]
    p["e_capoluogo"] = pd.Series(p.index, index=p.index).eq(p["capoluogo"])
    p["nome"] = [NOMI.get(i, i) for i in p.index]
    return p


p_lav_21, p_lav_11, p_std_11 = panel(2021, "lavoro"), panel(2011, "lavoro"), panel(2011, "studio")

# La distanza è in linea d'aria fra i centroidi ISTAT (EPSG:32633): non è distanza stradale
# né tempo di viaggio, e per i comuni montani la sottostima. Serve come controllo, non come
# misura di accessibilità.
print(f"Bagheria-Palermo, centroidi in linea d'aria: {p_lav_21.loc[BAGHERIA, 'km_capoluogo']:.1f} km")
print()
vicini_30 = (p_lav_21[p_lav_21["km_capoluogo"].le(30) & ~p_lav_21["e_capoluogo"]]
             .sort_values("pendolari", ascending=False))
print("I comuni entro 30 km da Palermo, ordinati per dimensione (lavoro 2021):")
print(vicini_30[["nome", "km_capoluogo", "pendolari", "quota_fuori"]].head(8).round(1).to_string(index=False))

Bagheria-Palermo, centroidi in linea d'aria: 17.5 km

I comuni entro 30 km da Palermo, ordinati per dimensione (lavoro 2021):
            nome  km_capoluogo  pendolari  quota_fuori
         Marsala          14.4    20396.0         12.4
          Modica          13.7    16797.0         16.8
        Vittoria          15.9    16320.0         15.3
        Acireale          20.8    13899.0         43.0
    Misterbianco           5.8    12739.0         66.2
          Alcamo          24.2    12691.0         24.3
        Bagheria          17.5    11974.0         40.2
Mazara del Vallo          24.5    11787.0         12.8


In [6]:
def residuo(p: pd.DataFrame, colonna: str = "quota_fuori", extra: str = "") -> dict:
    """Quanto Bagheria si discosta da un comune della stessa distanza e della stessa taglia.

    Modello volutamente povero — due regressori geografici/dimensionali, entrambi esogeni
    rispetto alla mobilità — perché serve a togliere di mezzo taglia e posizione, non a
    spiegare il fenomeno. Errori standard HC3: la varianza cresce con la dimensione.
    """
    d = p[~p["e_capoluogo"] & p["pendolari"].gt(0)].dropna(subset=[colonna]).copy()
    d["lkm"] = np.log(d["km_capoluogo"].clip(lower=1))
    d["lpend"] = np.log(d["pendolari"])
    m = smf.ols(f"Q('{colonna}') ~ lkm + lpend{extra}", data=d).fit(cov_type="HC3")
    r = m.resid
    return dict(osservato=d.loc[BAGHERIA, colonna], atteso=m.fittedvalues[BAGHERIA],
                residuo=r[BAGHERIA], z=r[BAGHERIA] / r.std(),
                percentile=100 * (r < r[BAGHERIA]).mean(), R2=m.rsquared, n=len(d),
                coef_km=m.params["lkm"], coef_taglia=m.params["lpend"])


tabella = pd.DataFrame({
    "2011 studio": residuo(p_std_11), "2011 lavoro": residuo(p_lav_11), "2021 lavoro": residuo(p_lav_21),
}).T
print(tabella.round(2).to_string())
print()
for etichetta, p in [("2011 studio", p_std_11), ("2011 lavoro", p_lav_11), ("2021 lavoro", p_lav_21)]:
    # Stesso universo del modello — i 381 non capoluogo — altrimenti i due percentili non
    # sono confrontabili e il "salto" misurerebbe in parte il cambio di denominatore.
    confrontabili = p[~p["e_capoluogo"] & p["pendolari"].gt(0)]
    grezzo = 100 * (confrontabili["quota_fuori"] < p.loc[BAGHERIA, "quota_fuori"]).mean()
    print(f"  {etichetta}: percentile grezzo {grezzo:4.0f}  ->  percentile del residuo "
          f"{tabella.loc[etichetta, 'percentile']:4.0f}")

             osservato  atteso  residuo     z  percentile    R2      n  coef_km  coef_taglia
2011 studio      16.35   17.12    -0.76 -0.06       51.18  0.43  381.0   -10.42        -8.80
2011 lavoro      43.20   40.19     3.01  0.21       60.37  0.29  381.0   -14.43        -4.98
2021 lavoro      40.15   42.04    -1.88 -0.13       48.29  0.31  381.0   -15.10        -5.72

  2011 studio: percentile grezzo   16  ->  percentile del residuo   51
  2011 lavoro: percentile grezzo   51  ->  percentile del residuo   60
  2021 lavoro: percentile grezzo   36  ->  percentile del residuo   48


**📌 Risultato chiave** — L'anomalia sparisce. A parità di distanza dal capoluogo e di
dimensione, Bagheria esce dal comune **quanto ci si aspetta**: residuo −1,9 punti nel 2021
(z = −0,13), +3,0 nel 2011, −0,8 sullo studio. In percentile il salto è netto: dal 36° grezzo
al 48° del residuo sul lavoro 2021, dal 16° al 51° sullo studio.

Entrambi i coefficienti hanno il segno atteso e sono fortemente significativi: più si è
lontani dal capoluogo, meno si esce (−15 punti per raddoppio della distanza); più si è
grandi, meno si esce (−5,7 punti per raddoppio della taglia).

> **Correzione a un claim del progetto.** La relazione riporta «Bagheria si muove poco —
> mobilità fuori comune `M2` al 25° percentile». Il percentile è esatto, la lettura no: è
> quello che ci si aspetta da un comune di 55.000 abitanti a 17 km dal capoluogo. La
> particolarità di Bagheria **non è quanto si muove**. È chi si muove, e per quale motivo.

# **4. Il ribaltamento: si esce per studiare, non per lavorare — e ha un genere**

Qui sta il risultato del thread. La misura è la stessa della sezione 3 — quota di chi esce
dal comune sul totale di chi si sposta per quel motivo — letta per genere.

Il denominatore è **già condizionato al motivo**: chi si sposta per lavoro un lavoro ce l'ha.
Lo scarto fra uomini e donne non è quindi un riflesso del divario occupazionale, che gli
altri due thread misurano altrove: è una misura indipendente sullo stesso passaggio.

E il conteggio del 2011 **non è una stima**: i record di tipo `S` della matrice sono
enumerazione esaustiva. Non c'è errore campionario da dichiarare, solo la scelta di misura.

In [7]:
def per_genere(anno: int, motivo: str, origini) -> pd.Series:
    d = od[od["anno"].eq(anno) & od["motivo"].eq(motivo)
           & od["origine"].isin(origini) & od["genere"].isin(["M", "F"])]
    g = d.pivot_table(index="genere", columns="luogo", values="persone", aggfunc="sum")
    return 100 * g["fuori"] / (g["dentro"] + g["fuori"])


def da_benchmark(territorio: str, motivo: str) -> pd.Series:
    b = benchmark[benchmark["anno"].eq(2011) & benchmark["territorio"].eq(territorio)
                  & benchmark["motivo"].eq(motivo)]
    g = b.pivot_table(index="genere", columns="luogo", values="persone", aggfunc="sum")
    return 100 * g["fuori"] / (g["dentro"] + g["fuori"])


righe = []
for etichetta, calcolo in [
    ("Bagheria", lambda m: per_genere(2011, m, [BAGHERIA])),
    ("Comune di Palermo", lambda m: per_genere(2011, m, [PALERMO])),
    ("5 comuni vicini", lambda m: per_genere(2011, m, VICINI)),
    ("Sicilia", lambda m: da_benchmark("ITG1", m)),
    ("Italia", lambda m: da_benchmark("IT", m)),
]:
    for motivo in ("studio", "lavoro"):
        q = calcolo(motivo)
        righe.append(dict(territorio=etichetta, motivo=motivo, F=q["F"], M=q["M"], gap_F_M=q["F"] - q["M"]))

confronto = pd.DataFrame(righe)
larga = confronto.pivot_table(index="territorio", columns="motivo", values="gap_F_M")
larga["ribaltamento"] = larga["studio"] - larga["lavoro"]
ordine = ["Bagheria", "Comune di Palermo", "5 comuni vicini", "Sicilia", "Italia"]
print("Quota che esce dal comune, per genere (2011, conteggio esaustivo):")
print(confronto.set_index(["territorio", "motivo"]).loc[ordine].round(1).to_string())
print()
print("Lo scarto F-M cambia segno fra studio e lavoro. L'ampiezza del salto:")
print(larga.loc[ordine].round(1).to_string())

Quota che esce dal comune, per genere (2011, conteggio esaustivo):
                             F     M  gap_F_M
territorio        motivo                     
Bagheria          studio  17.6  15.0      2.6
                  lavoro  35.2  47.2    -12.1
Comune di Palermo studio   0.7   0.8     -0.1
                  lavoro   4.7   6.7     -2.0
5 comuni vicini   studio  44.5  41.0      3.5
                  lavoro  62.8  64.6     -1.8
Sicilia           studio  18.8  17.4      1.4
                  lavoro  27.2  31.8     -4.6
Italia            studio  26.9  25.1      1.8
                  lavoro  43.3  48.0     -4.8

Lo scarto F-M cambia segno fra studio e lavoro. L'ampiezza del salto:
motivo             lavoro  studio  ribaltamento
territorio                                     
Bagheria            -12.1     2.6          14.7
Comune di Palermo    -2.0    -0.1           1.9
5 comuni vicini      -1.8     3.5           5.4
Sicilia              -4.6     1.4           6.0
Italia               -

In [8]:
# Dove sta Bagheria fra i 390, su queste tre misure? Stesso schema della sezione 3.
def gap_390(anno: int, motivo: str) -> pd.DataFrame:
    d = od[od["anno"].eq(anno) & od["motivo"].eq(motivo)
           & od["origine"].isin(COMUNI390) & od["genere"].isin(["M", "F"])]
    g = d.pivot_table(index=["origine", "genere"], columns="luogo", values="persone", aggfunc="sum").fillna(0)
    q = (100 * g["fuori"] / (g["dentro"] + g["fuori"])).unstack("genere")
    return q


g_std, g_lav = gap_390(2011, "studio"), gap_390(2011, "lavoro")
posizione = p_lav_11.copy()
posizione["gap_studio_F_M"] = g_std["F"] - g_std["M"]
posizione["gap_lavoro_F_M"] = g_lav["F"] - g_lav["M"]
posizione["ribaltamento"] = posizione["gap_studio_F_M"] - posizione["gap_lavoro_F_M"]

esiti = pd.DataFrame({c: residuo(posizione, c) for c in
                      ["gap_studio_F_M", "gap_lavoro_F_M", "ribaltamento"]}).T
esiti["percentile_grezzo"] = [100 * (posizione[c] < posizione.loc[BAGHERIA, c]).mean()
                              for c in esiti.index]
print(esiti[["osservato", "atteso", "residuo", "z", "percentile", "percentile_grezzo", "R2"]].round(2).to_string())

                osservato  atteso  residuo     z  percentile  percentile_grezzo    R2
gap_studio_F_M       2.62    1.03     1.59  0.30       66.93              52.31  0.03
gap_lavoro_F_M     -12.05   -3.56    -8.49 -1.10       13.91              14.87  0.00
ribaltamento        14.67    4.59    10.08  1.07       87.14              82.56  0.02


In [9]:
# Replica su una fonte che non condivide né tavola né anno né metodo di rilevazione:
# il censimento permanente 2018-2019, già lavorato dal thread genere.
permanente = pd.read_csv(PROCESSED / "genere_pendolarismo.csv")
permanente["gap_F_M"] = -permanente["gap_M_meno_F"]
replica = permanente.pivot_table(index=["nome_territorio", "anno"], columns="motivo", values="gap_F_M")
replica["ribaltamento"] = replica["STD"] - replica["WK"]
print("Censimento permanente 2018-2019 (STD = studio, WK = lavoro):")
print(replica.round(1).to_string())
print()
print("Confronto fra le due fonti sul ribaltamento, in punti percentuali:")
print(pd.DataFrame({
    "matrice OD 2011": larga.loc[["Bagheria", "Comune di Palermo", "Sicilia", "Italia"], "ribaltamento"],
    "cens. permanente 2019": replica.xs(2019, level="anno")["ribaltamento"].reindex(
        ["Bagheria", "Palermo", "Sicilia", "Italia"]).values,
}).round(1).to_string())

Censimento permanente 2018-2019 (STD = studio, WK = lavoro):
motivo                STD   WK  ribaltamento
nome_territorio anno                        
Bagheria        2018  3.0 -8.3          11.3
                2019  2.7 -8.2          10.9
Italia          2018  1.5 -5.0           6.5
                2019  1.4 -4.7           6.1
Palermo         2018  0.2 -1.7           1.9
                2019  0.1 -1.5           1.6
Sicilia         2018  1.7 -3.9           5.6
                2019  1.8 -4.1           5.9

Confronto fra le due fonti sul ribaltamento, in punti percentuali:
                   matrice OD 2011  cens. permanente 2019
territorio                                               
Bagheria                      14.7                   10.9
Comune di Palermo              1.9                    1.6
Sicilia                        6.0                    5.9
Italia                         6.5                    6.1


**📌 Risultato chiave** — Il segno dello scarto fra donne e uomini **cambia col motivo in
tutti i territori**: ovunque le ragazze escono per studiare più dei coetanei e le donne
escono per lavorare meno degli uomini. La particolarità di Bagheria è l'ampiezza del salto:
**+14,7 punti**, contro +6,0 in Sicilia, +6,6 in Italia, +1,9 nel Comune di Palermo. Due volte
e mezza il valore regionale.

Scomposto: sullo studio Bagheria è nella norma (+2,6 contro +1,4 siciliano). È **sul lavoro**
che si stacca: −12,1 punti contro −4,6 della Sicilia, cioè il **13° percentile** dei 381
comuni non capoluogo. Le ragazze di Bagheria si muovono; le donne di Bagheria no.

**La replica tiene.** La stessa misura sul censimento permanente 2018-2019 — altra
rilevazione, altro metodo, sette anni dopo — dà per Bagheria +11,3 e +10,9 contro +5,6/+5,9
siciliano: stesso ordine di grandezza, stesso rapporto col benchmark. Non è un artefatto del
2011.

**Attenzione all'`R²`**: i modelli della tabella spiegano quasi nulla (fra lo 0% e il 3%) della varianza
fra comuni. Il divario di genere nella mobilità è largamente idiosincratico, quindi «atteso»
qui vuol dire poco più che «media siciliana». Lo z di −1,10 va letto per quello che è —
Bagheria è circa una deviazione standard più sbilanciata del comune medio — non come una
misura di eccezionalità estrema.

# **5. Come ci si va: il treno è il canale femminile**

Mezzo, orario e durata stanno nei record di tipo `L` della matrice, e per i comuni sopra i
20.000 abitanti — Bagheria è uno — sono **rilevati su un campione**: sono stime, non conteggi.
Il leggimi ISTAT prescrive di usare il conteggio esaustivo per tutto ciò che sta nel tipo `S`
e la stima solo quando servono queste tre variabili. `pipeline/build.py` tiene le due cose in
due tabelle separate proprio per non farle mescolare.

Prima di leggere le quote, si misura quanto quella stima sia precisa. **Il file porta la
propria misura d'errore**: per gli stessi strati esistono sia il conteggio esaustivo sia la
stima, quindi lo scarto fra i due è osservabile e non va assunto.

In [10]:
esatto = (od[od["anno"].eq(2011) & od["origine"].eq(BAGHERIA)]
          .groupby(["genere", "motivo", "luogo"])["persone"].sum())
stimato = mezzo[mezzo["origine"].eq(BAGHERIA)].groupby(["genere", "motivo", "luogo"])["stima"].sum()
precisione = pd.DataFrame({"conteggio esaustivo": esatto, "stima campionaria": stimato}).dropna()
precisione["errore_%"] = 100 * (precisione["stima campionaria"] / precisione["conteggio esaustivo"] - 1)
print(precisione.round(2).to_string())
print(f"\nerrore relativo assoluto: mediano {precisione['errore_%'].abs().median():.2f}%, "
      f"massimo {precisione['errore_%'].abs().max():.2f}% (sullo strato più piccolo)")

                      conteggio esaustivo  stima campionaria  errore_%
genere motivo luogo                                                   
F      lavoro dentro                 2413            2446.07      1.37
              fuori                  1309            1303.16     -0.45
       studio dentro                 4344            4401.19      1.32
              fuori                   931             926.29     -0.51
M      lavoro dentro                 3922            3928.56      0.17
              fuori                  3509            3521.01      0.34
       studio dentro                 4382            4548.95      3.81
              fuori                   775             841.73      8.61

errore relativo assoluto: mediano 0.91%, massimo 8.61% (sullo strato più piccolo)


In [11]:
fuori_bag = mezzo[mezzo["origine"].eq(BAGHERIA) & mezzo["luogo"].eq("fuori")].copy()
fuori_bag["classe"] = fuori_bag["mezzo"].map(dict(zip(mezzi["mezzo"], mezzi["classe"])))
# Calibrazione ai margini esatti: ogni strato viene riportato al suo conteggio esaustivo.
# Se la conclusione dipendesse dall'errore campionario, qui cambierebbe.
fattore = (precisione["conteggio esaustivo"] / precisione["stima campionaria"])
fuori_bag["calibrata"] = fuori_bag["stima"] * pd.MultiIndex.from_frame(
    fuori_bag[["genere", "motivo", "luogo"]]).map(fattore)


def composizione(colonna: str) -> pd.DataFrame:
    tot = fuori_bag.groupby("genere")[colonna].sum()
    per_classe = fuori_bag.pivot_table(index="classe", columns="genere", values=colonna, aggfunc="sum")
    treno = fuori_bag[fuori_bag["mezzo"].eq("treno")].groupby("genere")[colonna].sum().rename("di cui treno")
    return (100 * pd.concat([per_classe, treno.to_frame().T]) / tot).round(1)


print("Come esce dal comune chi parte da Bagheria (2011, % sul totale di chi esce):")
print(composizione("stima").to_string())
print()
quota_treno = lambda c: (100 * fuori_bag[fuori_bag["mezzo"].eq("treno")].groupby("genere")[c].sum()
                         / fuori_bag.groupby("genere")[c].sum())
print("Quota treno, prima e dopo la calibrazione sui margini esatti:")
print(pd.DataFrame({"stima grezza": quota_treno("stima"),
                    "calibrata": quota_treno("calibrata")}).round(1).to_string())
g = quota_treno("calibrata")
print(f"\nscarto F-M sulla quota treno: {g['F'] - g['M']:+.1f} punti (calibrato)")

Come esce dal comune chi parte da Bagheria (2011, % sul totale di chi esce):
genere                     F     M
altro                    0.1   0.1
aziendale o scolastico   0.4   1.1
collettivo              34.7  19.5
piedi o bici             1.1   0.9
privato a motore        63.7  78.4
di cui treno            31.5  17.0

Quota treno, prima e dopo la calibrazione sui margini esatti:
        stima grezza  calibrata
genere                         
F               31.5       31.5
M               17.0       16.4

scarto F-M sulla quota treno: +15.1 punti (calibrato)


In [12]:
# E rispetto agli altri comuni siciliani? Stessa misura, stesso modello.
tutti_fuori = mezzo[mezzo["luogo"].eq("fuori")]
totale_comune = tutti_fuori.groupby("origine")["stima"].sum()
uso = pd.DataFrame({
    "treno": 100 * tutti_fuori[tutti_fuori["mezzo"].eq("treno")].groupby("origine")["stima"].sum()
             .reindex(totale_comune.index).fillna(0) / totale_comune,
    "oltre_30_min": 100 * tutti_fuori[tutti_fuori["durata"].isin(["31-60 min", "oltre 60 min"])]
                    .groupby("origine")["stima"].sum().reindex(totale_comune.index).fillna(0) / totale_comune,
})
posizione = posizione.join(uso)
print(pd.DataFrame({c: residuo(posizione, c) for c in ["treno", "oltre_30_min"]}).T
      [["osservato", "atteso", "residuo", "z", "percentile", "R2"]].round(2).to_string())

              osservato  atteso  residuo     z  percentile    R2
treno             21.89    4.03    17.87  2.50       97.38  0.05
oltre_30_min      47.74   38.05     9.69  0.62       73.75  0.14


**📌 Risultato chiave** — Due fatti, e il secondo ribalta l'ipotesi con cui il thread era
partito.

**Il treno di Bagheria funziona ed è usato.** Fra chi esce dal comune, il 21,9% prende il
treno: il **97° percentile** dei comuni siciliani, contro un atteso del 4,0%. Il collegamento
ferroviario con Palermo non è un'infrastruttura sottoutilizzata da promuovere — è già
l'asset di mobilità più distintivo che Bagheria abbia. Una proposta di policy costruita su
«portare il treno a Bagheria» sarebbe costruita su un problema che non esiste.

**Ma è un canale con un genere.** Fra le donne che escono da Bagheria il treno vale il 31,5%
degli spostamenti e il mezzo collettivo il 34,7%; fra gli uomini 17,0% e 19,5%. Gli uomini
guidano (78,4% mezzo privato a motore, contro 63,7%). La calibrazione ai margini esatti non
sposta la conclusione — anzi allarga lo scarto da 14,5 a 15,1 punti — e l'errore campionario
mediano è dello 0,9%.

Le donne di Bagheria che raggiungono Palermo lo fanno **sul mezzo collettivo, non in auto**.
Il che significa che per loro l'orario del servizio non è una comodità: è il vincolo.

# **6. Quando si parte**

La matrice dà la fascia oraria di uscita di casa. È l'unica variabile temporale disponibile —
del ritorno non si sa nulla, ed è un limite che pesa proprio sull'ipotesi del carico di cura.

In [13]:
orari = fuori_bag.pivot_table(index="orario", columns="genere", values="calibrata", aggfunc="sum")
ordine_orari = ["prima delle 7:15", "7:15-8:14", "8:15-9:14", "dopo le 9:14"]
print("Fascia oraria di uscita, chi esce da Bagheria (2011, % per genere):")
print((100 * orari / orari.sum()).loc[ordine_orari].round(1).to_string())
print()
solo_palermo = mezzo[mezzo["origine"].eq(BAGHERIA) & mezzo["verso_palermo"]]
for motivo in ("studio", "lavoro"):
    o = solo_palermo[solo_palermo["motivo"].eq(motivo)].pivot_table(
        index="orario", columns="genere", values="stima", aggfunc="sum")
    print(f"-- verso Palermo, per {motivo}")
    print((100 * o / o.sum()).loc[ordine_orari].round(1).to_string())
print()
durate = fuori_bag.pivot_table(index="durata", columns="genere", values="calibrata", aggfunc="sum")
print("Durata del tragitto (% per genere):")
print((100 * durate / durate.sum()).loc[["fino a 15 min", "16-30 min", "31-60 min", "oltre 60 min"]].round(1).to_string())

Fascia oraria di uscita, chi esce da Bagheria (2011, % per genere):
genere               F     M
orario                      
prima delle 7:15  44.7  61.1
7:15-8:14         39.0  27.9
8:15-9:14         10.5   4.7
dopo le 9:14       5.7   6.4

-- verso Palermo, per studio
genere               F     M
orario                      
prima delle 7:15  48.4  51.5
7:15-8:14         40.9  37.6
8:15-9:14          5.8   6.4
dopo le 9:14       4.9   4.5
-- verso Palermo, per lavoro
genere               F     M
orario                      
prima delle 7:15  54.4  65.0
7:15-8:14         33.7  24.5
8:15-9:14          6.4   3.2
dopo le 9:14       5.6   7.3

Durata del tragitto (% per genere):
genere            F     M
durata                   
fino a 15 min  15.0  15.0
16-30 min      32.6  39.8
31-60 min      43.8  36.1
oltre 60 min    8.7   9.1


**📌 Risultato chiave** — Chi va a lavorare a Palermo parte molto presto: il **65,0% degli
uomini** esce di casa prima delle 7:15, contro il **54,4% delle donne**. Le donne si
concentrano nelle due fasce successive (33,7% contro 24,5% fra le 7:15 e le 8:14; 6,4% contro
3,2% fra le 8:15 e le 9:14) e viaggiano più a lungo: il 43,8% impiega fra 31 e 60 minuti,
contro il 36,2% degli uomini, per un tragitto di 17 chilometri.

Sullo studio la differenza fra i due generi quasi scompare (48,4% contro 51,5% nella prima
fascia). **La divergenza oraria nasce col lavoro**, come tutto il resto di questo thread.

# **7. Un meccanismo che i dati non sostengono**

L'ipotesi naturale a questo punto è: se il mezzo collettivo è il canale delle donne, allora
dove il mezzo collettivo pesa di più il divario di genere dovrebbe essere più piccolo. È
verificabile sui 390 comuni, quindi si verifica — e il risultato va riportato comunque venga.

In [14]:
uso_completo = uso.join(pd.DataFrame({
    "collettivo": 100 * tutti_fuori.assign(
        classe=tutti_fuori["mezzo"].map(dict(zip(mezzi["mezzo"], mezzi["classe"]))))
        .query("classe == 'collettivo'").groupby("origine")["stima"].sum()
        .reindex(totale_comune.index).fillna(0) / totale_comune}))
prova = posizione.join(uso_completo["collettivo"]).dropna(
    subset=["gap_lavoro_F_M", "collettivo", "treno", "km_capoluogo", "pendolari"])
prova = prova[~prova["e_capoluogo"]].copy()
prova["lkm"] = np.log(prova["km_capoluogo"].clip(lower=1))
prova["lpend"] = np.log(prova["pendolari"])

print(f"n = {len(prova)} comuni non capoluogo\n")
for v in ("collettivo", "treno"):
    rho, p = stats.spearmanr(prova[v], prova["gap_lavoro_F_M"])
    m = smf.ols(f"gap_lavoro_F_M ~ {v} + lkm + lpend", data=prova).fit(cov_type="HC3")
    print(f"{v:11}: Spearman rho = {rho:+.3f} (p = {p:.2f})   "
          f"coefficiente condizionato {m.params[v]:+.3f} p.p. per punto di quota (p = {m.pvalues[v]:.2f})")
print()
prova["quartile"] = pd.qcut(prova["collettivo"], 4, labels=["Q1 meno collettivo", "Q2", "Q3", "Q4 più collettivo"])
print(prova.groupby("quartile", observed=True).agg(
    comuni=("gap_lavoro_F_M", "size"), quota_collettivo_mediana=("collettivo", "median"),
    gap_F_M_mediano=("gap_lavoro_F_M", "median")).round(1).to_string())

n = 381 comuni non capoluogo

collettivo : Spearman rho = -0.096 (p = 0.06)   coefficiente condizionato -0.076 p.p. per punto di quota (p = 0.10)


treno      : Spearman rho = +0.055 (p = 0.29)   coefficiente condizionato +0.040 p.p. per punto di quota (p = 0.38)

                    comuni  quota_collettivo_mediana  gap_F_M_mediano
quartile                                                             
Q1 meno collettivo      96                      13.5             -2.6
Q2                      95                      20.5             -3.5
Q3                      95                      25.4             -2.8
Q4 più collettivo       95                      32.2             -4.8


**📌 Risultato negativo, riportato** — L'ipotesi **non regge**. Sui 390 comuni la quota di
mezzo collettivo non è associata a un divario di genere più piccolo: il rho di Spearman è
−0,10 (p = 0,06) e il segno è quello sbagliato, il quartile con più mezzo collettivo ha il
divario *più ampio* (−4,8 contro −2,6). Sul treno l'associazione è nulla (p = 0,29).

Conseguenza per la proposta: **il vincolo non è l'offerta di trasporto**. Non si può
sostenere, con questi dati, che aumentare corse o linee riduca il divario di genere nella
mobilità — e infatti la sezione 5 mostra che a Bagheria il treno è già al 97° percentile.
Questo esclude l'intervento infrastrutturale come leva principale e sposta il bersaglio sul
passaggio studio→lavoro, che è dove il divario si apre.

È un risultato che restringe lo spazio delle proposte difendibili, ed è per questo che sta
nel notebook invece che nel cestino.

# **8. L'ultimo miglio a Palermo — dati del Comune di Palermo**

La terza fonte del bando. Chi arriva in treno da Bagheria scende a Palermo Centrale e deve
ancora raggiungere il posto di lavoro o l'università: se quel tratto non è servito nelle
fasce che servono, il treno non basta. Si misura sul **GTFS di AMAT** pubblicato dal Comune
di Palermo — il file grezzo è quello già scaricato dal thread educazione, che qui si legge
e non si tocca.

In [15]:
import zipfile

GTFS = RADICE / "data" / "raw" / "edu" / "palermo_gtfs_2026-08-25.zip"
with zipfile.ZipFile(GTFS) as z:
    stops = pd.read_csv(z.open("stops.txt"), dtype=str)
    stop_times = pd.read_csv(z.open("stop_times.txt"), dtype=str)
    trips = pd.read_csv(z.open("trips.txt"), dtype=str)
    calendario = pd.read_csv(z.open("calendar_dates.txt"), dtype=str)

centrale = stops[stops["stop_name"].str.contains("STAZIONE CENTRALE", case=False, na=False)]
attivi = calendario[calendario["exception_type"].eq("1")]
# Giorno con più corse programmate: il feed è di agosto, quindi è servizio estivo ridotto.
giorno = (trips.merge(attivi, on="service_id").groupby("date").size().idxmax())
servizi = set(attivi.loc[attivi["date"].eq(giorno), "service_id"])
corse = trips[trips["service_id"].isin(servizi)]

passaggi = stop_times[stop_times["stop_id"].isin(centrale["stop_id"])].merge(
    corse[["trip_id", "route_id"]], on="trip_id")
per_ora = passaggi["departure_time"].str.slice(0, 2).astype(int).value_counts().sort_index()

print(f"feed AMAT, giorno più servito: {giorno} — {len(corse):,} corse in rete")
print(f"fermate 'Stazione Centrale': {len(centrale)}   linee che le servono: {passaggi['route_id'].nunique()}")
print(f"passaggi totali alla Centrale nel giorno: {len(passaggi):,}\n")
for ora, n in per_ora.items():
    if 5 <= ora <= 22:
        print(f"  {ora:02d}:00-{ora:02d}:59  {n:4d}  {'▇' * int(n / 4)}")

feed AMAT, giorno più servito: 20260731 — 6,516 corse in rete
fermate 'Stazione Centrale': 11   linee che le servono: 18
passaggi totali alla Centrale nel giorno: 2,296

  05:00-05:59    34  ▇▇▇▇▇▇▇▇
  06:00-06:59   101  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  07:00-07:59   126  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  08:00-08:59   127  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  09:00-09:59   128  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  10:00-10:59   128  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  11:00-11:59   130  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  12:00-12:59   128  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  13:00-13:59   129  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  14:00-14:59   128  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  15:00-15:59   130  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  16:00-16:59   130  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  17:00-17:59   125  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  18:00-18:59   126  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  19:00-19:59   128  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
  20:00-20:59   136  ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇

**📌 Risultato chiave** — L'ultimo miglio a Palermo **non è il collo di bottiglia**: le
fermate «Stazione Centrale» sono servite da 18 linee con circa 125-130 passaggi l'ora,
continuativamente dalle 7 alle 21. Chi scende dal treno trova rete urbana.

*Limite*: il feed disponibile è di agosto, cioè servizio estivo ridotto — il dato di settembre
sarebbe più alto, non più basso, quindi la conclusione regge a maggior ragione. E la rete
AMAT è **urbana**: non copre la tratta Bagheria-Palermo, che è ferroviaria e di competenza
regionale.

# **9. Il bersaglio, in persone**

Il template della proposal chiede evidenza → intervento → target → KPI. Qui si produce il
**target**: quante donne, in numeri, separano Bagheria dal benchmark. Nessuna di queste cifre
va scritta a mano altrove: si rigenera da questa cella.

In [16]:
bag_lav = od[od["anno"].eq(2011) & od["origine"].eq(BAGHERIA) & od["motivo"].eq("lavoro")
             & od["genere"].isin(["M", "F"])]
conti = bag_lav.pivot_table(index="genere", columns="luogo", values="persone", aggfunc="sum")
conti["totale"] = conti.sum(axis=1)
conti["quota_fuori"] = 100 * conti["fuori"] / conti["totale"]
print(conti.round(1).to_string())
print()

gap_sicilia = larga.loc["Sicilia", "lavoro"]
scenari = {
    "parità con gli uomini di Bagheria": conti.loc["M", "quota_fuori"],
    "divario di genere pari a quello siciliano": conti.loc["M", "quota_fuori"] + gap_sicilia,
}
platea_f = conti.loc["F", "totale"]
for nome, obiettivo in scenari.items():
    delta = platea_f * (obiettivo - conti.loc["F", "quota_fuori"]) / 100
    print(f"  {nome:44} quota obiettivo {obiettivo:4.1f}%  ->  +{delta:3.0f} donne fuori comune")
print(f"\n(su {platea_f:,.0f} donne di Bagheria che si spostano per lavoro, 2011)")
print()
print("Per confronto, gli scenari occupazionali del thread genere (2024):")
print(pd.read_csv(PROCESSED / "genere_gap_persone.csv").to_string(index=False))

luogo   dentro  fuori  totale  quota_fuori
genere                                    
F         2413   1309    3722         35.2
M         3922   3509    7431         47.2

  parità con gli uomini di Bagheria            quota obiettivo 47.2%  ->  +449 donne fuori comune
  divario di genere pari a quello siciliano    quota obiettivo 42.7%  ->  +279 donne fuori comune

(su 3,722 donne di Bagheria che si spostano per lavoro, 2011)

Per confronto, gli scenari occupazionali del thread genere (2024):
                                scenario  occupate in più (2024)  occupate in più (media 2022-2024)
parità con i coetanei maschi di Bagheria                     239                                221
              tasso femminile di Palermo                      40                                 34
             tasso femminile dell'Italia                     262                                255


**📌 Risultato chiave** — Portare le pendolari di Bagheria al divario **medio siciliano** —
non alla parità, al semplice comportamento regionale — significa **+279 donne** che lavorano
fuori comune. La parità piena con gli uomini di Bagheria ne varrebbe 449.

L'ordine di grandezza è coerente con gli scenari occupazionali del thread genere (+239 con la
parità coi coetanei, +262 col tasso femminile italiano): è lo stesso gruppo di persone visto
da due misure che non condividono né tavola né denominatore.

# **10. Cosa questo thread non afferma**

1. **Nessuna età.** La matrice del pendolarismo non ha la dimensione età, e nemmeno la tavola
   del censimento permanente (verificato: `AGE_NOCLASS` è servita solo come `TOTAL`, anche a
   livello nazionale). Il target 15-34 del bando **non è isolabile** su questi dati. Quello
   che si può dire è che *il motivo* è un'informazione d'età parziale: chi esce dal comune per
   studio è quasi solo secondaria superiore e università, perché nido, materna, primaria e
   secondaria di primo grado a Bagheria ci sono. Il ribaltamento studio→lavoro è quindi un
   passaggio che avviene *dentro* la finestra 15-34, ma non è misurato per età.
2. **Nessuna causalità.** I confronti fra comuni sono ecologici: orientano, non dimostrano. Il
   modello della sezione 3 serve a togliere di mezzo taglia e distanza, non a spiegare la
   mobilità, e il suo `R²` sui divari di genere è vicino a zero.
3. **Nessuna serie 2011→2021 sui livelli.** Le due matrici usano definizioni diverse
   (*giornalmente* contro *almeno tre giorni a settimana*) e il 2021 copre il solo lavoro e non
   ha il sesso. Si confronta la composizione, mai il livello, e la colonna `definizione` lo
   porta scritto riga per riga.
4. **Nessun dato sul ritorno.** La matrice dà l'orario di uscita di casa, non quello di
   rientro. L'ipotesi del carico di cura resta un'ipotesi: i dati mostrano che le donne partono
   più tardi e viaggiano più a lungo, non perché.
5. **Nessun effetto del trasporto sul divario.** Testato in sezione 7, non trovato.
6. **Il 2011 sui mezzi è una stima.** Conteggio esaustivo per origine, destinazione, sesso e
   motivo; campione per mezzo, orario e durata. Le due cose stanno in due tabelle separate e
   la precisione della stima è misurata in sezione 5, non assunta.

# **Le tavole per le figure**
Python scrive, R legge. Nessuna logica di trasformazione negli script R.

In [17]:
uscite = {}

# fig14 — dove vanno i pendolari di Bagheria, con le coordinate per la mappa a flussi.
flussi_mappa = []
for anno, motivo in [(2011, "studio"), (2011, "lavoro"), (2021, "lavoro")]:
    t = flussi(anno, motivo).reset_index()
    t["anno"], t["motivo"] = anno, motivo
    flussi_mappa.append(t)
flussi_mappa = pd.concat(flussi_mappa)
flussi_mappa = flussi_mappa.merge(centroidi.rename(columns={"territorio": "destinazione",
                                                            "x": "x_dest", "y": "y_dest"})
                                  [["destinazione", "x_dest", "y_dest"]], on="destinazione", how="left")
flussi_mappa["x_orig"] = centroidi.set_index("territorio").loc[BAGHERIA, "x"]
flussi_mappa["y_orig"] = centroidi.set_index("territorio").loc[BAGHERIA, "y"]
uscite["mob_flussi_bagheria.csv"] = flussi_mappa

# fig15 — il ribaltamento, cinque territori, due motivi, due generi.
uscite["mob_ribaltamento.csv"] = confronto
# La forma larga: la differenza fra i due scarti è calcolata qui, non in R
# (nessuna logica di trasformazione negli script di viz/).
uscite["mob_ribaltamento_territori.csv"] = (
    larga.loc[ordine].reset_index()
    .rename(columns={"studio": "gap_studio_F_M", "lavoro": "gap_lavoro_F_M"}))
uscite["mob_ribaltamento_390.csv"] = (posizione[["nome", "gap_studio_F_M", "gap_lavoro_F_M",
                                                 "ribaltamento", "pendolari", "km_capoluogo"]]
                                      .reset_index().rename(columns={"origine": "territorio"}))

# fig16 — mezzo e orario per genere, più la posizione di Bagheria sui 390.
composizione_mezzo = (fuori_bag.groupby(["genere", "classe"])["calibrata"].sum().rename("persone")
                      .reset_index())
composizione_mezzo["quota"] = 100 * composizione_mezzo["persone"] / composizione_mezzo.groupby(
    "genere")["persone"].transform("sum")
treno_riga = (fuori_bag[fuori_bag["mezzo"].eq("treno")].groupby("genere")["calibrata"].sum()
              .rename("persone").reset_index().assign(classe="di cui: treno"))
treno_riga["quota"] = 100 * treno_riga["persone"] / fuori_bag.groupby("genere")["calibrata"].sum().values
uscite["mob_mezzo_genere.csv"] = pd.concat([composizione_mezzo, treno_riga], ignore_index=True)
uscite["mob_orario_genere.csv"] = (fuori_bag.groupby(["genere", "orario"])["calibrata"].sum()
    .rename("persone").reset_index()
    .assign(quota=lambda d: 100 * d["persone"] / d.groupby("genere")["persone"].transform("sum")))
# Il denominatore va nella tavola perché senza di lui la quota inganna: a Lampedusa e
# Linosa esce dal comune UNA persona, che prende il treno, e il comune risulta al 100%.
# Le figure escludono i comuni sotto le 100 uscite, e lo dichiarano.
posizione["pendolari_fuori"] = tutti_fuori.groupby("origine")["stima"].sum()
uscite["mob_treno_390.csv"] = (posizione[["nome", "treno", "oltre_30_min", "quota_fuori",
                                          "km_capoluogo", "pendolari", "pendolari_fuori"]]
                               .reset_index().rename(columns={"origine": "territorio"}))

# fig17 — il panel della sezione 3 sull'anno più recente (2021), con l'atteso del modello.
d = p_lav_21[~p_lav_21["e_capoluogo"] & p_lav_21["pendolari"].gt(0)].copy()
d["lkm"] = np.log(d["km_capoluogo"].clip(lower=1))
d["lpend"] = np.log(d["pendolari"])
modello = smf.ols("quota_fuori ~ lkm + lpend", data=d).fit(cov_type="HC3")
d["atteso"] = modello.fittedvalues
d["residuo"] = modello.resid
uscite["mob_taglia_distanza.csv"] = (d[["nome", "quota_fuori", "atteso", "residuo",
                                        "km_capoluogo", "pendolari", "verso_palermo", "fuori"]]
                                     .reset_index().rename(columns={"origine": "territorio"}))

# L'atteso NON è funzione della sola distanza: dipende anche dalla taglia, quindi in figura
# non si può tirare una linea attraverso i punti (verrebbe una spezzata senza significato).
# Qui si producono due curve a taglia fissata — il comune mediano e Bagheria — che è anche
# il modo di far vedere che è la taglia a spostare la curva in basso.
griglia = np.geomspace(d["km_capoluogo"].min(), d["km_capoluogo"].max(), 60)
curve = []
for etichetta, pendolari in [("comune mediano", d["pendolari"].median()),
                             ("taglia di Bagheria", d.loc[BAGHERIA, "pendolari"])]:
    previsione = modello.predict(pd.DataFrame({"lkm": np.log(np.clip(griglia, 1, None)),
                                               "lpend": np.log(pendolari)}))
    curve.append(pd.DataFrame({"scenario": etichetta, "pendolari": pendolari,
                               "km_capoluogo": griglia, "atteso": previsione.values}))
uscite["mob_curva_attesa.csv"] = pd.concat(curve, ignore_index=True)

# la sintesi che la proposal cita
# I percentili delle note si calcolano, non si scrivono: erano quattro cifre a mano e
# tre erano fuori posto (il 91 non riproducibile, il treno troncato invece che arrotondato,
# il percentile del divario senza denominatore, che e' la stessa ambiguita' gia' sanata a
# valle nei documenti). `d` sono i 381 non capoluogo con pendolari, `posizione` i 390.
pc_palermo_lav = 100 * (100 * d["verso_palermo"] / d["fuori"] <
                        100 * d.loc[BAGHERIA, "verso_palermo"] / d.loc[BAGHERIA, "fuori"]).mean()
pc_treno = 100 * (posizione["treno"] < posizione.loc[BAGHERIA, "treno"]).mean()
pc_gap_lav = 100 * (posizione["gap_lavoro_F_M"] < posizione.loc[BAGHERIA, "gap_lavoro_F_M"]).mean()

uscite["mob_sintesi.csv"] = pd.DataFrame([
    dict(misura="quota che esce dal comune per lavoro, 2021", valore=p_lav_21.loc[BAGHERIA, "quota_fuori"],
         unita="%", nota="a parità di taglia e distanza: nella media (z=-0,13)"),
    dict(misura="quota di chi esce che va a Palermo, lavoro 2021",
         valore=100 * p_lav_21.loc[BAGHERIA, "verso_palermo"] / p_lav_21.loc[BAGHERIA, "fuori"],
         unita="%", nota=f"{pc_palermo_lav:.0f}° percentile dei {len(d)} comuni non capoluogo"),
    dict(misura="quota di chi esce che va a Palermo, studio 2011",
         valore=100 * p_std_11.loc[BAGHERIA, "verso_palermo"] / p_std_11.loc[BAGHERIA, "fuori"],
         unita="%", nota="93° percentile"),
    dict(misura="divario F-M sull'uscire per lavoro, 2011", valore=larga.loc["Bagheria", "lavoro"],
         unita="p.p.", nota=f"Sicilia -4,6 · {pc_gap_lav:.0f}° percentile sui {len(posizione)} "
                              f"(13° sui 381 non capoluogo) · replicato nel 2018-2019"),
    dict(misura="divario F-M sull'uscire per studio, 2011", valore=larga.loc["Bagheria", "studio"],
         unita="p.p.", nota="Sicilia +1,4 · le ragazze escono più dei coetanei"),
    dict(misura="ribaltamento studio-lavoro, 2011", valore=larga.loc["Bagheria", "ribaltamento"],
         unita="p.p.", nota="Sicilia +6,0 · Italia +6,5 · Palermo +1,9"),
    dict(misura="quota treno fra chi esce, donne", valore=quota_treno("calibrata")["F"],
         unita="%", nota=f"uomini 16,4% · comune nel complesso al {pc_treno:.0f}° "
                          f"percentile siciliano"),
    dict(misura="donne in più fuori comune col divario siciliano",
         valore=platea_f * (conti.loc["M", "quota_fuori"] + gap_sicilia - conti.loc["F", "quota_fuori"]) / 100,
         unita="persone", nota="bersaglio del KPI; parità piena = 449"),
])

for nome, tavola_out in uscite.items():
    tavola_out.to_csv(PROCESSED / nome, index=False)
    print(f"  {nome:32} {len(tavola_out):>6,} righe")
print()
print(uscite["mob_sintesi.csv"].round(1).to_string(index=False))

  mob_flussi_bagheria.csv             201 righe
  mob_ribaltamento.csv                 10 righe
  mob_ribaltamento_territori.csv        5 righe
  mob_ribaltamento_390.csv            390 righe
  mob_mezzo_genere.csv                 12 righe
  mob_orario_genere.csv                 8 righe
  mob_treno_390.csv                   390 righe
  mob_taglia_distanza.csv             381 righe
  mob_curva_attesa.csv                120 righe
  mob_sintesi.csv                       8 righe

                                         misura  valore   unita                                                                                        nota
     quota che esce dal comune per lavoro, 2021    40.2       %                                        a parità di taglia e distanza: nella media (z=-0,13)
quota di chi esce che va a Palermo, lavoro 2021    65.1       %                                                 97° percentile dei 381 comuni non capoluogo
quota di chi esce che va a Palermo, studio 2011    

---

## In una riga

Il pendolarismo verso Palermo **è misurabile**, ed è quasi tutto quello che Bagheria fa fuori
dai propri confini. Bagheria non si muove poco: si muove quanto un comune della sua taglia a
quella distanza. Quello che non fa è **portare le donne dentro quel flusso**: le ragazze
escono per studiare più dei coetanei, le donne escono per lavorare 12 punti meno degli
uomini, e chi ci riesce lo fa in treno. Il vincolo non è l'infrastruttura — è già usata più
che ovunque in Sicilia — ma il passaggio in cui quella mobilità si perde.